# openEO save_result Zarr Integration Test
**Backend:** https://dev.openeo.eurac.edu/openeo/1.1.0/
**Purpose:** Test robustness of the EURAC dev backend:
- OIDC auth via EURAC Keycloak
- List collections, inspect Sentinel-2 sample (B04+B08 for NDVI)
- NDVI → save_result to Zarr via `execute_batch` (one-liner) **and** manual `create_job` → `start_job` → poll → `get_results`
- `load_stac` with EMO1 Zarr collection
**Known constraints:**
- Backend only declares `netCDF` as output format — we test whether `save_result` with `format='Zarr'` still works


In [1]:
import openeo
import json
import time
import os
import tarfile
import xarray as xr
import numpy as np
from IPython.display import display, JSON

## 1. Connect & Authenticate

In [2]:
BACKEND_URL = "https://dev.openeo.eurac.edu/openeo/1.1.0/"
conn = openeo.connect(BACKEND_URL)
print("Backend:", conn.capabilities().url)
print("API version:", conn.capabilities().api_version_check)

Backend: https://dev.openeo.eurac.edu/openeo/1.1.0/
API version: 1.1.0


In [3]:
conn.authenticate_oidc()
print("Authenticated:", conn.auth is not None)

Authenticated using refresh token.
Authenticated: True


## 2. List & Inspect Collections

In [4]:
collections = conn.list_collections()
print(f"Total: {len(collections)}")
for c in collections:
    title = c.get('title', '') or c.get('description', '')[:60] if c.get('description') else ''
    print(f"  - {c['id']}: {title}")

Total: 129
  - ADO_CORINE_100m_3035: Corine Land Cover (CLC) 2018
  - ADO_EU_DEM_25m_3035: Copernicus Land Monitoring Service - EU-DEM 
  - ADO_EVAP_ET_MOD16_500m_3035: MOD16 Evapotranspiration - 500 m
  - ADO_EVAP_SSEBOP_1km_4326: SSEBop Evapotranspiration - 1 km
  - ADO_factor_available_water_capacity: Factor available water capacity
  - ADO_factor_distance_to_water: Factor distance to water
  - ADO_factor_elevation: Factor elevation
  - ADO_factor_landscape_diversity: Factor landscape diversity
  - ADO_factor_organic_carbon_content: Factor humus content
  - ADO_factor_presence_of_irrigation_infrastructure: Factor presence of irrigation infrastructure
  - ADO_factor_slope: Factor slope
  - ADO_factor_soil_texture: Factor soil texture
  - ADO_LST_MODIS_231m_3035: Land Surface Temperature - 231m 8 day mean
  - ADO_NDVI_MODIS_231m_3035: Normalized Difference Vegetation Index - 231m 8 day Maximum Value Composite
  - ADO_REL_RR_12_ERA5_QM: Precipitation Anomalies - ERA5_QM REL_RR-12
  - A

## 3. Batch Job: execute_batch (one-liner)

`execute_batch` creates → starts → polls → downloads the job in one call.
We pass `out_format='Zarr'` so it auto-adds a `save_result(format='Zarr')` node.

In [5]:
cube = conn.load_collection(
    'SENTINEL2_L2A_SAMPLE',
    spatial_extent={'west': 11.341, 'south': 46.488, 'east': 11.355, 'north': 46.497},
    temporal_extent=['2022-06-02', '2022-06-30'],
    bands=['B04', 'B08']
)
ndvi = cube.ndvi(red='B04', nir='B08')

print("Process graph (without save_result — will be auto-added):")
display(JSON(ndvi.flat_graph()))

Process graph (without save_result — will be auto-added):


<IPython.core.display.JSON object>

In [6]:
OUTPUT_DIR = "/tmp/openeo_zarr_test"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Launching execute_batch with out_format='Zarr'...")
print("This creates the job, starts it, polls until completion, and downloads results.")

job = ndvi.execute_batch(
    outputfile=OUTPUT_DIR,
    out_format='Zarr',
    print=print,
)

print(f"\nJob ID: {job.job_id}")
print("Downloaded files:")
for f in os.listdir(OUTPUT_DIR):
    print(f"  - {f}")

Launching execute_batch with out_format='Zarr'...
This creates the job, starts it, polls until completion, and downloads results.
0:00:00 Job '5f9ced61-fdf2-4c1d-9e84-9a077694598f': send 'start'
0:00:00 Job '5f9ced61-fdf2-4c1d-9e84-9a077694598f': queued (progress N/A)
0:00:05 Job '5f9ced61-fdf2-4c1d-9e84-9a077694598f': queued (progress N/A)
0:00:11 Job '5f9ced61-fdf2-4c1d-9e84-9a077694598f': running (progress N/A)
0:00:19 Job '5f9ced61-fdf2-4c1d-9e84-9a077694598f': running (progress N/A)
0:00:29 Job '5f9ced61-fdf2-4c1d-9e84-9a077694598f': running (progress N/A)
0:00:41 Job '5f9ced61-fdf2-4c1d-9e84-9a077694598f': finished (progress N/A)

Job ID: 5f9ced61-fdf2-4c1d-9e84-9a077694598f
Downloaded files:
  - 20220602000000_20220630000000_data
  - extracted


## 5. Inspect Downloaded Result

The result was downloaded as a tar archive (Zarr is a directory, served as `.tar` by the API).
Extract and open the Zarr contents via xarray.

In [7]:
import tarfile
import xarray as xr

OUTPUT_DIR = "/tmp/openeo_zarr_test"
tar_path = os.path.join(OUTPUT_DIR, "20220602000000_20220630000000_data")

# Extract the tar archive
extract_dir = os.path.join(OUTPUT_DIR, "extracted")
os.makedirs(extract_dir, exist_ok=True)
with tarfile.open(tar_path, "r") as tar:
    tar.extractall(path=extract_dir)
    print("Extracted files:")
    for m in tar.getmembers():
        print(f"  - {m.name}")

# Open the Zarr store directly via xarray
zarr_path = os.path.join(extract_dir, "None.zarr")
ds = xr.open_zarr(zarr_path)
print(ds)

# Inspect the NDVI data variable
print("\n=== Data Variable (name) ===")
data = ds["name"].values
non_nan = data[~np.isnan(data)]
print(f"Shape (time, lat, lon): {data.shape}")
print(f"Valid pixels: {len(non_nan)} / {data.size}")
print(f"Range: [{float(non_nan.min()):.4f}, {float(non_nan.max()):.4f}]")
print(f"Mean: {float(non_nan.mean()):.4f}")
print(f"Sample values: {[float(v) for v in non_nan[:5]]}")

Extracted files:
  - None.zarr
  - None.zarr/latitude
  - None.zarr/latitude/c
  - None.zarr/latitude/c/0
  - None.zarr/latitude/zarr.json
  - None.zarr/longitude
  - None.zarr/longitude/c
  - None.zarr/longitude/c/0
  - None.zarr/longitude/zarr.json
  - None.zarr/name
  - None.zarr/name/c
  - None.zarr/name/c/0
  - None.zarr/name/c/0/0
  - None.zarr/name/c/0/0/0
  - None.zarr/name/c/1
  - None.zarr/name/c/1/0
  - None.zarr/name/c/1/0/0
  - None.zarr/name/c/10
  - None.zarr/name/c/10/0
  - None.zarr/name/c/10/0/0
  - None.zarr/name/c/11
  - None.zarr/name/c/11/0
  - None.zarr/name/c/11/0/0
  - None.zarr/name/c/2
  - None.zarr/name/c/2/0
  - None.zarr/name/c/2/0/0
  - None.zarr/name/c/3
  - None.zarr/name/c/3/0
  - None.zarr/name/c/3/0/0
  - None.zarr/name/c/4
  - None.zarr/name/c/4/0
  - None.zarr/name/c/4/0/0
  - None.zarr/name/c/5
  - None.zarr/name/c/5/0
  - None.zarr/name/c/5/0/0
  - None.zarr/name/c/6
  - None.zarr/name/c/6/0
  - None.zarr/name/c/6/0/0
  - None.zarr/name/c/7
  - N

/tmp/ipykernel_928851/3809919941.py:11: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_dir)


<xarray.Dataset> Size: 1MB
Dimensions:      (time: 12, latitude: 81, longitude: 132)
Coordinates:
  * time         (time) datetime64[ns] 96B 2022-06-02 2022-06-05 ... 2022-06-30
  * latitude     (latitude) float64 648B 46.5 46.5 46.5 ... 46.49 46.49 46.49
  * longitude    (longitude) float64 1kB 11.34 11.34 11.34 ... 11.35 11.35 11.35
    spatial_ref  int32 4B ...
Data variables:
    name         (time, latitude, longitude) float64 1MB dask.array<chunksize=(1, 81, 132), meta=np.ndarray>
Attributes:
    openeo_x_dim:          longitude
    openeo_y_dim:          latitude
    openeo_temporal_dims:  ['time']
    openeo_band_dims:      ['bands']
    openeo_other_dims:     []

=== Data Variable (name) ===
Shape (time, lat, lon): (12, 81, 132)
Valid pixels: 118452 / 128304
Range: [0.0000, 184.0787]
Mean: 2.6305
Sample values: [0.042198233562315994, 0.04279390063944909, 0.04648862512363996, 0.03175378602833415, 0.03175378602833415]


In [8]:
import xarray as xr
ds = xr.open_dataset('/tmp/openeo_zarr_test/extracted/None.zarr', engine='zarr')
ds

<xarray.Dataset> Size: 1MB
Dimensions:      (time: 12, latitude: 81, longitude: 132)
Coordinates:
  * time         (time) datetime64[ns] 96B 2022-06-02 2022-06-05 ... 2022-06-30
  * latitude     (latitude) float64 648B 46.5 46.5 46.5 ... 46.49 46.49 46.49
  * longitude    (longitude) float64 1kB 11.34 11.34 11.34 ... 11.35 11.35 11.35
    spatial_ref  int32 4B ...
Data variables:
    name         (time, latitude, longitude) float64 1MB ...
Attributes:
    openeo_x_dim:          longitude
    openeo_y_dim:          latitude
    openeo_temporal_dims:  ['time']
    openeo_band_dims:      ['bands']
    openeo_other_dims:     []

## 6. Output Formats & Capabilities

In [9]:
caps = conn.capabilities().capabilities
print("Declared output formats:")
for fmt, info in caps.get('output_formats', {}).items():
    print(f"  - {fmt}: {info.get('title', info)}")

print("\nEndpoints supporting POST (job-related):")
for ep in caps.get('endpoints', []):
    if 'POST' in ep['methods']:
        print(f"  {ep['path']} {ep['methods']}")

Declared output formats:
  - GTiff: GTiff
  - COG: COG
  - netCDF: netCDF
  - Zarr: Zarr

Endpoints supporting POST (job-related):
  /jobs ['POST']
  /jobs/{job_id} ['POST']
  /jobs/{job_id}/results ['POST']


## Summary

| What | Status | Notes |
|------|--------|-------|
| OIDC auth | ✅ | Device code flow via EURAC Keycloak |
| Collections listing | ✅ | 138 collections |
| `create_job` (batch) | ✅ | Fixed by pydantic v2 + auth boundary cleanup |
| `execute_batch` (create + start + download) | ✅ | Fixed: AnyUrl Hera crash, missing spec skip, dask-gateway→LocalCluster, Zarr dir as tar |
| `save_result` with Zarr | ✅ | Zarr output + download via tar archive works |
